In [1]:
from pykalshi import KalshiClient, Action, Side
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

load_dotenv()

client = KalshiClient()

In [3]:
from pykalshi import MarketStatus, History, AsyncHistory
from decimal import Decimal

markets = client.get_markets(series_ticker='KXNHLGAME')

market = min(markets, key=lambda m: Decimal(m.volume_fp or "0"))

print(len(markets))

market = markets[23]

market

100


Ticker,KXNHLGAME-26MAY14MTLBUF-BUF
Title,Game 5: Montreal at Buffalo Winner?
Status,finalized
YES,$0.0000 / $0.0100
Last,$0.0100
Volume,2717576.66
Open Int,2008712.68
Result,NO
Closes,"May 15, 01:49"


In [2]:
from pykalshi import MarketStatus, History, AsyncHistory
from decimal import Decimal

markets = client.history.get_markets(series_ticker='KXNHLGAME')

market = min(markets, key=lambda m: Decimal(m.volume_fp or "0"))

print(len(markets))

market = markets[23]

market

100


Ticker,KXNHLGAME-26MAR24COLPIT-COL
Title,Colorado at Pittsburgh Winner?
Status,finalized
YES,$0.9900 / $1.0000
Last,$0.9900
Volume,215447.00
Open Int,0.00
Result,YES
Closes,"Mar 25, 02:20"


In [6]:
from pykalshi import CandlestickPeriod
from datetime import datetime, timedelta

def iso_to_ts(s: str) -> int:
    return int(datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp())

end = iso_to_ts(market.close_time)
start = end - int(timedelta(days=0.2).total_seconds())

candles = market.get_candlesticks(start_ts=start, end_ts=end, period=CandlestickPeriod.ONE_MINUTE)


candles_df = candles.to_dataframe()

candles_df


,ticker,end_period_ts,volume_fp,open_interest_fp,open_dollars,high_dollars,low_dollars,close_dollars,mean_dollars,timestamp
0,KXNHLGAME-26MAR18PHIANA-ANA,1773884580,4007.00,71623.00,0.5900,0.5900,0.5900,0.5900,0.5900,2026-03-19 01:43:00
1,KXNHLGAME-26MAR18PHIANA-ANA,1773884640,301.00,71924.00,0.5900,0.5900,0.5900,0.5900,0.5900,2026-03-19 01:44:00
2,KXNHLGAME-26MAR18PHIANA-ANA,1773884700,424.00,72348.00,0.5900,0.5900,0.5900,0.5900,0.5900,2026-03-19 01:45:00
3,KXNHLGAME-26MAR18PHIANA-ANA,1773884760,52.00,72400.00,0.5900,0.5900,0.5900,0.5900,0.5900,2026-03-19 01:46:00
4,KXNHLGAME-26MAR18PHIANA-ANA,1773884820,216.00,72591.00,0.5900,0.5900,0.5800,0.5900,0.5888,2026-03-19 01:47:00
...,...,...,...,...,...,...,...,...,...,...
181,KXNHLGAME-26MAR18PHIANA-ANA,1773895440,81628.00,709805.00,0.0200,0.1800,0.0200,0.1700,0.0595,2026-03-19 04:44:00
182,KXNHLGAME-26MAR18PHIANA-ANA,1773895500,129952.00,755443.00,0.1700,0.5200,0.0100,0.0100,0.2478,2026-03-19 04:45:00
183,KXNHLGAME-26MAR18PHIANA-ANA,1773895560,42419.00,781066.00,0.0100,0.0200,0.0100,0.0100,0.0101,2026-03-19 04:46:00
184,KXNHLGAME-26MAR18PHIANA-ANA,1773896100,0.00,781066.00,NaN,NaN,NaN,NaN,NaN,2026-03-19 04:55:00


In [7]:
# @title
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = candles.to_dataframe()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.7, 0.3],
)

# Candlesticks
fig.add_trace(
    go.Candlestick(
        x=df["timestamp"],
        open=df["open_dollars"].astype(float),
        high=df["high_dollars"].astype(float),
        low=df["low_dollars"].astype(float),
        close=df["close_dollars"].astype(float),
        increasing_line_color="#22c55e",
        decreasing_line_color="#ef4444",
        name="Price",
    ),
    row=1, col=1,
)

# Volume bars — colored by direction
colors = [
    "#22c55e" if c >= o else "#ef4444"
    for c, o in zip(df["close_dollars"].astype(float), df["open_dollars"].astype(float))
]
fig.add_trace(
    go.Bar(
        x=df["timestamp"],
        y=df["volume_fp"].astype(float),
        marker_color=colors,
        opacity=0.5,
        name="Volume",
    ),
    row=2, col=1,
)

fig.update_layout(
    title=f"{market.ticker}  —  Daily",
    template="plotly_dark",
    height=500,
    showlegend=False,
    xaxis_rangeslider_visible=False,
    margin=dict(l=50, r=20, t=50, b=30),
    yaxis=dict(title="Price ($)", side="right"),
    yaxis2=dict(title="Volume", side="right"),
)

fig.show()

In [8]:
def clean_candles_df(candles_df: pd.DataFrame) -> pd.DataFrame:
    df = candles_df.copy()

    price_cols = [
        "open_dollars",
        "high_dollars",
        "low_dollars",
        "close_dollars",
        "mean_dollars",
    ]

    # Convert price columns to numeric
    for col in price_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows where close is missing
    # close is required for returns / realized volatility
    df = df.dropna(subset=["close_dollars"])

    # Fill missing OHLC/mean values using close
    for col in ["open_dollars", "high_dollars", "low_dollars", "mean_dollars"]:
        df[col] = df[col].fillna(df["close_dollars"])

    # Sort data
    df = df.sort_values(["ticker", "end_period_ts"]).reset_index(drop=True)

    return df    

cleaned_candles_df = clean_candles_df(candles_df)

In [9]:
def iso_to_ts(s: str) -> int:
    return int(datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp())

def add_surface_axes(
    candles_df: pd.DataFrame,
    close_time: str,
    price_col: str = "close_dollars",
    eps: float = 1e-6,
) -> pd.DataFrame:
    df = candles_df.copy()

    close_ts = iso_to_ts(close_time)

    # Convert price to numeric first
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    # Drop rows where price is missing
    df = df.dropna(subset=[price_col])

    # Price is probability-like value
    df["price"] = df[price_col].clip(lower=eps, upper=1 - eps)

    # Odds and log-odds
    df["odds"] = df["price"] / (1 - df["price"])
    df["log_odds"] = np.log(df["odds"])

    # Time until market close, in days
    df["time_to_close_minutes"] = (close_ts - df["end_period_ts"]) / (60)

    df = df[
        (df["time_to_close_minutes"] > 0)
        & np.isfinite(df["log_odds"])
    ].copy()

    return df

In [10]:
surface_df = add_surface_axes(candles_df, market.close_time)

surface_df[[
    "ticker",
    "end_period_ts",
    "price",
    "odds",
    "log_odds",
    "time_to_close_minutes",
]].head()

,ticker,end_period_ts,price,odds,log_odds,time_to_close_minutes
0,KXNHLGAME-26MAR18PHIANA-ANA,1773884580,0.59,1.439024,0.363965,287.283333
1,KXNHLGAME-26MAR18PHIANA-ANA,1773884640,0.59,1.439024,0.363965,286.283333
2,KXNHLGAME-26MAR18PHIANA-ANA,1773884700,0.59,1.439024,0.363965,285.283333
3,KXNHLGAME-26MAR18PHIANA-ANA,1773884760,0.59,1.439024,0.363965,284.283333
4,KXNHLGAME-26MAR18PHIANA-ANA,1773884820,0.59,1.439024,0.363965,283.283333


In [11]:
import numpy as np
import pandas as pd

def add_realized_volatility(
    surface_df: pd.DataFrame,
    ticker_col: str = "ticker",
    ts_col: str = "end_period_ts",
    value_col: str = "log_odds",
    rolling_window: int = 10,
) -> pd.DataFrame:
    df = surface_df.copy()

    # Sort before calculating diffs
    df = df.sort_values([ticker_col, ts_col]).reset_index(drop=True)

    # Return in log-odds space
    df["log_odds_return"] = df.groupby(ticker_col)[value_col].diff()

    # Rolling realized volatility by ticker
    df["realized_vol"] = (
        df.groupby(ticker_col)["log_odds_return"]
        .rolling(rolling_window)
        .std()
        .reset_index(level=0, drop=True)
    )

    # Estimate candles per day from timestamps
    median_step_seconds = df.groupby(ticker_col)[ts_col].diff().median()

    if pd.isna(median_step_seconds) or median_step_seconds <= 0:
        periods_per_day = 24
    else:
        periods_per_day = int((24 * 60 * 60) / median_step_seconds)

    # Scale to daily realized volatility
    df["realized_vol_daily"] = df["realized_vol"] * np.sqrt(periods_per_day)

    df = df.dropna(subset=["log_odds_return", "realized_vol_daily"])

    return df

In [12]:
vol_df = add_realized_volatility(
    surface_df,
    rolling_window=10,
)

vol_df[[
    "ticker",
    "end_period_ts",
    "price",
    "log_odds",
    "time_to_close_minutes",
    "log_odds_return",
    "realized_vol_daily",
]].head()

,ticker,end_period_ts,price,log_odds,time_to_close_minutes,log_odds_return,realized_vol_daily
10,KXNHLGAME-26MAR18PHIANA-ANA,1773885180,0.59,0.363965,277.283333,0.000000,0.000000
11,KXNHLGAME-26MAR18PHIANA-ANA,1773885240,0.59,0.363965,276.283333,0.000000,0.000000
12,KXNHLGAME-26MAR18PHIANA-ANA,1773885300,0.59,0.363965,275.283333,0.000000,0.000000
13,KXNHLGAME-26MAR18PHIANA-ANA,1773885360,0.59,0.363965,274.283333,0.000000,0.000000
14,KXNHLGAME-26MAR18PHIANA-ANA,1773885420,0.58,0.322773,273.283333,-0.041192,0.494304


In [13]:
import numpy as np
import pandas as pd

def add_realized_volatility(
    surface_df: pd.DataFrame,
    ticker_col: str = "ticker",
    ts_col: str = "end_period_ts",
    value_col: str = "log_odds",
    rolling_window: int = 10,
) -> pd.DataFrame:
    df = surface_df.copy()

    df[ts_col] = pd.to_numeric(df[ts_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

    df = df.dropna(subset=[ticker_col, ts_col, value_col])
    df = df.sort_values([ticker_col, ts_col]).reset_index(drop=True)

    # Change in log-odds from previous candle
    df["log_odds_return"] = df.groupby(ticker_col)[value_col].diff()

    # Rolling realized volatility
    df["realized_vol"] = (
        df.groupby(ticker_col)["log_odds_return"]
        .rolling(rolling_window)
        .std()
        .reset_index(level=0, drop=True)
    )

    df = df.dropna(subset=["realized_vol"]).copy()

    return df

In [14]:
vol_df = add_realized_volatility(
    surface_df,
    rolling_window=10,
)

vol_df[[
    "ticker",
    "end_period_ts",
    "price",
    "log_odds",
    "time_to_close_minutes",
    "log_odds_return",
    "realized_vol",
]].head()

,ticker,end_period_ts,price,log_odds,time_to_close_minutes,log_odds_return,realized_vol
10,KXNHLGAME-26MAR18PHIANA-ANA,1773885180,0.59,0.363965,277.283333,0.000000,0.000000
11,KXNHLGAME-26MAR18PHIANA-ANA,1773885240,0.59,0.363965,276.283333,0.000000,0.000000
12,KXNHLGAME-26MAR18PHIANA-ANA,1773885300,0.59,0.363965,275.283333,0.000000,0.000000
13,KXNHLGAME-26MAR18PHIANA-ANA,1773885360,0.59,0.363965,274.283333,0.000000,0.000000
14,KXNHLGAME-26MAR18PHIANA-ANA,1773885420,0.58,0.322773,273.283333,-0.041192,0.013026


In [15]:
import plotly.graph_objects as go

def plot_realized_vol_points(vol_df):
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=vol_df["log_odds"],
                y=vol_df["time_to_close_minutes"],
                z=vol_df["realized_vol"],
                mode="markers",
                marker=dict(
                    size=3,
                    opacity=0.75,
                    color=vol_df["realized_vol"],
                    colorbar=dict(title="Realized Vol"),
                ),
                text=[
                    f"Ticker: {ticker}<br>"
                    f"Price: {price:.4f}<br>"
                    f"Log Odds: {log_odds:.4f}<br>"
                    f"Time to Close: {t:.1f} min<br>"
                    f"Realized Vol: {rv:.6f}"
                    for ticker, price, log_odds, t, rv in zip(
                        vol_df["ticker"],
                        vol_df["price"],
                        vol_df["log_odds"],
                        vol_df["time_to_close_minutes"],
                        vol_df["realized_vol"],
                    )
                ],
                hoverinfo="text",
            )
        ]
    )

    fig.update_layout(
        title="Realized Volatility Points",
        scene=dict(
            xaxis_title="Log Odds",
            yaxis_title="Time Till Close (minutes)",
            zaxis_title="Realized Volatility",
        ),
        width=1000,
        height=750,
    )

    fig.show()

In [16]:
plot_realized_vol_points(vol_df)

In [84]:
import numpy as np
from scipy.interpolate import griddata
import plotly.graph_objects as go

def plot_realized_vol_surface(vol_df, grid_size=60):
    x = vol_df["log_odds"].values
    y = vol_df["time_to_close_minutes"].values
    z = vol_df["realized_vol"].values

    x_grid = np.linspace(x.min(), x.max(), grid_size)
    y_grid = np.linspace(y.min(), y.max(), grid_size)

    X, Y = np.meshgrid(x_grid, y_grid)

    Z = griddata(
        points=(x, y),
        values=z,
        xi=(X, Y),
        method="linear",
    )

    Z_nearest = griddata(
        points=(x, y),
        values=z,
        xi=(X, Y),
        method="nearest",
    )

    Z = np.where(np.isnan(Z), Z_nearest, Z)

    fig = go.Figure(
        data=[
            go.Surface(
                x=X,
                y=Y,
                z=Z,
            )
        ]
    )

    fig.update_layout(
        title="Realized Volatility Surface",
        scene=dict(
            xaxis_title="Log Odds",
            yaxis_title="Time Till Close (minutes)",
            zaxis_title="Realized Volatility",
        ),
        width=1000,
        height=750,
    )

    fig.show()

In [85]:
plot_realized_vol_surface(vol_df)

In [4]:
import requests
import pandas as pd

game_id = 401283399

url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary"
params = {"event": game_id}

data = requests.get(url, params=params, timeout=20).json()

plays_raw = data["plays"]

plays = pd.json_normalize(plays_raw)

print(plays.columns)
print(plays[["text", "period.number", "clock.displayValue", "wallclock"]].head())

Index(['id', 'sequenceNumber', 'text', 'awayScore', 'homeScore', 'scoringPlay',
       'scoreValue', 'participants', 'wallclock', 'shootingPlay',
       'pointsAttempted', 'shortDescription', 'type.id', 'type.text',
       'period.number', 'period.displayValue', 'clock.displayValue', 'team.id',
       'coordinate.x', 'coordinate.y'],
      dtype='object')
                                                text  period.number  \
0  Isaiah Roby vs. Jonas Valanciunas (Grayson All...              1   
1     Jonas Valanciunas misses 7-foot two point shot              1   
2                    Darius Bazley defensive rebound              1   
3                        Dillon Brooks personal foul              1   
4    Darius Bazley misses 26-foot three point jumper              1   

  clock.displayValue             wallclock  
0              12:00  2021-02-18T02:10:06Z  
1              11:37  2021-02-18T02:10:25Z  
2              11:34  2021-02-18T02:10:28Z  
3              11:33  2021-02-18T02

In [7]:
plays[['wallclock', 'period.number', 'clock.displayValue']]

,wallclock,period.number,clock.displayValue
0,2021-02-18T02:10:06Z,1,12:00
1,2021-02-18T02:10:25Z,1,11:37
2,2021-02-18T02:10:28Z,1,11:34
3,2021-02-18T02:10:33Z,1,11:33
4,2021-02-18T02:11:05Z,1,11:24
...,...,...,...
467,2021-02-18T04:26:00Z,4,14.7
468,2021-02-18T04:26:10Z,4,6.6
469,2021-02-18T04:26:13Z,4,4.3
470,2021-02-18T04:26:20Z,4,0.0
